## Satellite-Style Field Map with Soil Texture Overlay

This visualization creates a realistic-looking map with:
- Satellite/aerial background (OpenStreetMap)
- Field boundaries colored by soil surface texture
- Field ID labels
- Legend showing soil type colors

**Note:** The NRCS soil data APIs require specific authentication or are currently unavailable via public REST. The soil data generated here is representative of typical Nebraska soil series in the region.

Fields included:
- field_001: Ely (Silt loam)
- field_002: Colo (Silty clay loam)
- field_003: Colo (Silty clay loam)
- field_004: Kennebec (Silt loam)
- field_005: Colo (Silty clay loam)

In [ ]:
import matplotlib.pyplot as plt
import contextily as ctx
import geopandas as gpd
import pandas as pd
from pathlib import Path
from shapely.geometry import Polygon
import numpy as np
from matplotlib.patches import Patch

DATA_DIR = Path('data')
FIELDS_DIR = DATA_DIR / 'fields'
SOIL_DIR = DATA_DIR / 'soil'

# Load field boundaries
fields_list = []
for geojson_file in sorted(FIELDS_DIR.glob('*.geojson')):
    gdf = gpd.read_file(geojson_file)
    fields_list.append(gdf)

# Combine all fields into one GeoDataFrame
fields_gdf = gpd.GeoDataFrame(pd.concat(fields_list, ignore_index=True))

# Load soil data
soil_data = []
for soil_file in sorted(SOIL_DIR.glob('*_soil.csv')):
    # Skip the combined file
    if soil_file.name == 'all_nebraska_fields_soil.csv':
        continue
    df = pd.read_csv(soil_file, comment='#')
    soil_data.append(df)

soil_df = pd.concat(soil_data, ignore_index=True)

# Merge soil data with field boundaries
fields_gdf = fields_gdf.merge(soil_df[['field_id', 'surface_texture']], on='field_id')

# Define color mapping for soil textures
# Using earth-tone colors that work well with satellite imagery
soil_colors = {
    'Silt loam': '#D2B48C',           # Tan
    'Silty clay loam': '#A0522D',     # Sienna
    'Silty clay': '#8B4513',          # SaddleBrown
    'Clay': '#654321',                # Dark brown
    'Loam': '#BC8F8F',                # RosyBrown
    'Sandy loam': '#F4A460',          # SandyBrown
    'Clay loam': '#8B0000',           # DarkRed
    'Muck': '#2F4F4F',                # DarkSlateGray
    'Sand': '#DEB887'                 # BurlyWood
}

# Create a color column for plotting
fields_gdf['color'] = fields_gdf['surface_texture'].map(soil_colors)
# Default color for unknown textures
fields_gdf['color'] = fields_gdf['color'].fillna('#808080')  # Gray

# Create the plot
fig, ax = plt.subplots(1, 1, figsize=(14, 12))

# Add OpenStreetMap satellite background
try:
    ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.HOT, zoom=15)
except Exception as e:
    print(f"Warning: Could not add basemap: {e}")
    print("Continuing without background...")
    # Set a light gray background if no basemap
    ax.set_facecolor('#f0f0f0')

# Plot fields with soil texture colors
fields_gdf.plot(
    ax=ax,
    color=fields_gdf['color'],
    edgecolor='black',
    linewidth=1.5,
    alpha=0.7
)

# Add field ID labels
for idx, row in fields_gdf.iterrows():
    # Get centroid for label placement
    centroid = row.geometry.centroid
    ax.annotate(
        text=row['field_id'],
        xy=(centroid.x, centroid.y),
        xytext=(0, 0),
        textcoords='offset points',
        ha='center',
        va='center',
        fontsize=10,
        fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='black')
    )

# Customize the plot
ax.set_title('Nebraska Agricultural Fields\nSurface Soil Types Overlaid on Satellite Imagery', 
            fontsize=16, fontweight='bold', pad=20)
ax.set_axis_off()  # Hide axes for cleaner look

# Create legend for soil types
legend_elements = []
for texture, color in soil_colors.items():
    if texture in fields_gdf['surface_texture'].values:
        legend_elements.append(Patch(facecolor=color, edgecolor='black', label=texture))

if legend_elements:
    ax.legend(handles=legend_elements, loc='lower left', title='Soil Surface Texture', 
              bbox_to_anchor=(0, 0), fontsize=10, title_fontsize=12)

# Adjust layout and display
plt.tight_layout()
plt.show()

# Print summary info
print(f"Generated map with {len(fields_gdf)} fields")
print("Field IDs and their soil textures:")
for idx, row in fields_gdf.iterrows():
    print(f"  {row['field_id']}: {row['surface_texture']}")
